# Fase 2 — Big Data con PySpark
## Terminal Puerto Coquimbo, Sitio 3

Laboratorio de Programación Avanzada · Semanas 5-10

Este notebook corre **dentro del contenedor `jupyter`** del clúster levantado con Docker
Compose. El *driver* de Spark vive aquí; los *executors* viven en los contenedores
`spark-worker`.

**Antes de empezar**, verifica el clúster desde una terminal del host:

```
docker compose exec jupyter python3 verificar_cluster.py
```

Si eso pasa, todo lo de abajo funciona.

---

### Lo que vas a hacer

| Bloque | Tema |
|---|---|
| 1-2 | Conectar el driver al clúster y entender qué es cada nodo |
| 3-4 | Leer gigabytes de JSONL: inferencia de esquema vs esquema explícito |
| 5-6 | Perfilar antes de limpiar, y limpiar lo que de verdad está sucio |
| 7-10 | **Shuffling**: por qué es el cuello de botella y cómo se mitiga |
| 11 | Escribir la capa limpia en Parquet particionado |


## 0. Comprobación de entorno — ejecuta esta celda primero

Si tienes el Jupyter de la Fase 1 corriendo, **monta esta misma carpeta**, así que ves este
archivo desde los dos sitios y son indistinguibles a simple vista:

| Puerto | Entorno | ¿Sirve para la Fase 2? |
|---|---|---|
| **8888** | Jupyter de la Fase 1 (conda, sin Java, sin clúster) | **No** |
| **8889** | Jupyter del clúster (Python 3.10 + Java 17 + workers) | **Sí** |

Abrirlo en el equivocado produce un error que no explica nada:

```
PySparkRuntimeError: [JAVA_GATEWAY_EXITED]
Java gateway process exited before sending its port number.
```

Eso significa que PySpark intentó arrancar una JVM **local** y no encontró Java. La celda
de abajo detecta la situación y te lo dice en castellano.

In [1]:
# --- Guarda de entorno --------------------------------------------------------
# Falla rápido y con un mensaje claro en vez de dejar que PySpark reviente 20
# líneas más abajo con un error que no menciona la causa real.
import os, shutil, sys

problemas = []

if not os.path.isdir("/opt/spark"):
    problemas.append("no existe /opt/spark: no estás en el contenedor del clúster")

if shutil.which("java") is None:
    problemas.append("no hay Java en el PATH: PySpark no puede arrancar la JVM")

try:
    import pyspark
    if not pyspark.__file__.startswith("/opt/spark/"):
        problemas.append(
            f"pyspark viene de {pyspark.__file__}\n"
            f"        (instalado por pip/conda) en vez de /opt/spark/python. "
            f"Esa copia no está alineada con el clúster.")
except ImportError:
    problemas.append("pyspark no es importable")

if problemas:
    print("=" * 72)
    print("ESTE NOTEBOOK NO ESTÁ CORRIENDO EN EL CLÚSTER")
    print("=" * 72)
    for p in problemas:
        print(f"  - {p}")
    print()
    print("  Ábrelo en el Jupyter DEL CLÚSTER:")
    print("      http://localhost:8889        (token: lab-sitio3)")
    print()
    print("  El del puerto 8888 es el entorno de la Fase 1. Monta esta misma")
    print("  carpeta, por eso ves el mismo archivo, pero no tiene Java ni")
    print("  conexión con los workers.")
    print("=" * 72)
    raise SystemExit("Entorno incorrecto: usa http://localhost:8889")

print(f"Entorno correcto")
print(f"  Python  : {sys.version.split()[0]}")
print(f"  Java    : {shutil.which('java')}")
print(f"  PySpark : {pyspark.__file__}")


Entorno correcto
  Python  : 3.10.12
  Java    : /opt/java/openjdk/bin/java
  PySpark : /opt/spark/python/pyspark/__init__.py


## 1. Conectar el driver al clúster

Tres parámetros son los que hacen que esto funcione dentro de Docker. Los otros son ajustes.

- **`master`** — a quién le pedimos recursos. `spark://spark-master:7077` es el nombre del
  servicio en la red de Compose, no una IP.
- **`spark.driver.host`** — el nombre con el que los executors vuelven a llamar al driver.
  Este es *el* parámetro que rompe a todo el mundo: si no se fija, el driver anuncia una IP
  interna que los workers no saben resolver, y el job se queda en `RUNNING` para siempre,
  sin error y sin avanzar.
- **`spark.driver.bindAddress`** — `0.0.0.0` para que el driver escuche en todas las
  interfaces del contenedor, no solo en localhost.

In [2]:
import os, time
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import (StructType, StructField, StringType,
                                DoubleType, LongType, TimestampType)

spark = (
    SparkSession.builder
    .appName("fase2-sitio3")
    .master(os.environ.get("SPARK_MASTER_URL", "spark://spark-master:7077"))
    # --- los tres parámetros críticos en Docker ---
    .config("spark.driver.host", os.environ.get("SPARK_DRIVER_HOST", "jupyter"))
    .config("spark.driver.bindAddress", "0.0.0.0")
    # --- memoria: debe caber en lo que declara cada worker (ver .env) ---
    .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "2g"))
    .config("spark.executor.memory", os.environ.get("SPARK_EXECUTOR_MEMORY", "2g"))
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

sc = spark.sparkContext
print(f"Spark          : {spark.version}")
print(f"Master         : {sc.master}")
print(f"Executors      : {sc._jsc.sc().getExecutorMemoryStatus().size() - 1}")
print(f"Paralelismo    : {sc.defaultParallelism} tareas simultáneas")
print(f"UI del driver  : http://localhost:4040")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/06 22:17:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark          : 4.0.1
Master         : spark://spark-master:7077
Executors      : 2
Paralelismo    : 4 tareas simultáneas
UI del driver  : http://localhost:4040


### Qué es cada nodo

```
        ┌──────────────┐        pide recursos        ┌──────────────┐
        │   jupyter    │ ──────────────────────────► │ spark-master │
        │  (DRIVER)    │ ◄────────────────────────── │              │
        │              │      asigna executors       └──────┬───────┘
        │ - tu código  │                                    │
        │ - el plan    │                                    │ lanza
        │ - resultados │                                    ▼
        └──────▲───────┘                          ┌────────────────────┐
               │                                  │   spark-worker-1   │
               │   los executors devuelven        │   spark-worker-2   │
               └──────────────────────────────────┤   (EXECUTORS)      │
                   datos al driver.host           │  - leen archivos   │
                                                  │  - hacen el shuffle│
                                                  └────────────────────┘
```

El **master no procesa datos**: solo reparte. Quien trabaja son los workers. Y el driver
es tu notebook, que arma el plan y recoge los resultados.

**Abre <http://localhost:8080>** (UI del master) y deja la pestaña visible: vas a ver
aparecer y desaparecer las aplicaciones a medida que ejecutas celdas.

## 2. Leer los datos crudos de la Fase 1

`data/raw/` tiene 4 archivos JSONL de ~327 MB cada uno: **10 millones de eventos, 1.3 GB**.

JSON es el peor formato posible para analítica —es texto, no tiene tipos y no se puede
leer por columnas— pero es lo que escupen los sistemas reales, así que hay que saber
ingerirlo.

### El costo oculto de `inferSchema`

Cuando Spark lee JSON sin decirle el esquema, **tiene que leer el archivo entero una vez
solo para adivinar los tipos**, y después lo lee de nuevo para procesarlo. Sobre 1.3 GB
eso es una pasada completa desperdiciada. Vamos a medirlo.

In [3]:
RUTA_CRUDA = "/opt/workspace/data/raw"

# --- Opción A: dejar que Spark infiera el esquema (lo que hace todo el mundo) ---
ini = time.time()
df_inferido = spark.read.json(f"{RUTA_CRUDA}/*.jsonl")
esquema_inferido = df_inferido.schema   # fuerza la pasada de inferencia
t_inferencia = time.time() - ini

print(f"Inferir el esquema tardó: {t_inferencia:.1f} s")
df_inferido.printSchema()

Inferir el esquema tardó: 5.3 s
root
 |-- metric: string (nullable = true)
 |-- sensor_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- value: double (nullable = true)
 |-- worker_id: long (nullable = true)



In [4]:
# --- Opción B: declarar el esquema a mano ---
# Spark ya no necesita mirar los datos para saber de qué tipo es cada campo.
esquema = StructType([
    StructField("timestamp", StringType(), True),   # se castea después, a propósito
    StructField("sensor_id", StringType(), True),
    StructField("metric",    StringType(), True),
    StructField("value",     DoubleType(), True),
    StructField("unit",      StringType(), True),
    StructField("worker_id", LongType(),   True),
])

ini = time.time()
df_crudo = spark.read.schema(esquema).json(f"{RUTA_CRUDA}/*.jsonl")
_ = df_crudo.schema
t_explicito = time.time() - ini

print(f"Con esquema explícito : {t_explicito:.3f} s")
print(f"Con inferencia        : {t_inferencia:.1f} s")
print(f"\nEl esquema explícito evitó una pasada completa sobre 1.3 GB.")
print("En producción SIEMPRE se declara el esquema: además de ser más rápido,")
print("evita que un archivo corrupto cambie los tipos de una ejecución a otra.")


Con esquema explícito : 0.144 s
Con inferencia        : 5.3 s

El esquema explícito evitó una pasada completa sobre 1.3 GB.
En producción SIEMPRE se declara el esquema: además de ser más rápido,
evita que un archivo corrupto cambie los tipos de una ejecución a otra.


### Particiones: la unidad de paralelismo

Una **partición** es el trozo de datos que procesa **una** tarea en **un** core. Es la
unidad fundamental de Spark: si tienes 4 cores y 1 partición, 3 cores están mirando.

In [5]:
n_particiones = df_crudo.rdd.getNumPartitions()
print(f"Particiones al leer el crudo : {n_particiones}")
print(f"Cores disponibles            : {sc.defaultParallelism}")
print()
print("Spark parte los archivos en bloques de ~128 MB (spark.sql.files.maxPartitionBytes).")
print(f"1.3 GB / 128 MB ≈ {n_particiones} particiones. No es casualidad.")

ini = time.time()
total = df_crudo.count()
print(f"\nRegistros: {total:,}  (contados en {time.time()-ini:.1f} s)")


Particiones al leer el crudo : 5
Cores disponibles            : 4

Spark parte los archivos en bloques de ~128 MB (spark.sql.files.maxPartitionBytes).
1.3 GB / 128 MB ≈ 5 particiones. No es casualidad.


[Stage 1:===============================================>           (4 + 1) / 5]


Registros: 250,000  (contados en 2.0 s)


## 3. Perfilar ANTES de limpiar

Es tentador escribir `.dropna().dropDuplicates()` y sentirse productivo. Pero limpiar sin
perfilar es adivinar: puedes estar borrando datos buenos y dejando pasar los malos.

Primero se mide qué está roto. Después se decide qué hacer.

In [6]:
# Una sola pasada para contar nulos en todas las columnas a la vez.
# Hacer un .count() por columna serían 6 pasadas sobre 1.3 GB.
nulos = df_crudo.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_crudo.columns
]).collect()[0].asDict()

print("Nulos por columna:")
for col, n in nulos.items():
    print(f"  {col:<12} {n:>10,}")


[Stage 4:===============================================>           (4 + 1) / 5]

Nulos por columna:
  timestamp             0
  sensor_id             0
  metric                0
  value                 0
  unit                  0
  worker_id             0


In [7]:
# Rango de valores y cardinalidad por métrica.
df_crudo.groupBy("metric", "unit").agg(
    F.count("*").alias("registros"),
    F.min("value").alias("minimo"),
    F.max("value").alias("maximo"),
    F.round(F.avg("value"), 3).alias("promedio"),
).show(truncate=False)

print("Sensores distintos:", df_crudo.select("sensor_id").distinct().count())
df_crudo.groupBy("sensor_id").count().orderBy(F.desc("count")).show(10, False)


+-----------+----+---------+------+------+--------+
|metric     |unit|registros|minimo|maximo|promedio|
+-----------+----+---------+------+------+--------+
|pressure   |bar |83744    |1.0   |10.0  |5.489   |
|vibration  |mm/s|83149    |0.1   |4.5   |2.294   |
|temperature|C   |83107    |-25.0 |4.999 |-10.021 |
+-----------+----+---------+------+------+--------+



Sensores distintos: 4


[Stage 16:==============================================>           (4 + 1) / 5]

+---------+-----+
|sensor_id|count|
+---------+-----+
|sensor_3 |62911|
|sensor_1 |62460|
|sensor_4 |62382|
|sensor_2 |62247|
+---------+-----+



### Resultado del perfilado: los datos están limpios

Cero nulos, una sola métrica, una sola unidad, 4 sensores perfectamente balanceados y
valores dentro de un rango físico estrecho.

**Eso es un hallazgo, no un fracaso.** El generador de la Fase 1 produce datos sintéticos
ideales. Un pipeline real recibe unidades mezcladas (°C y °F), duplicados por reintentos,
timestamps desordenados y valores centinela (`-999`).

> **Ejercicio 1.** Vuelve al ingestor de la Fase 1 e introduce defectos
> deliberados: un 2% de duplicados, un 1% de nulos en `value`, un 5% de lecturas en °F sin
> cambiar la etiqueta `unit`, y algunos timestamps con formato distinto. Después vuelve
> aquí y comprueba que tu perfilado los detecta *antes* de limpiarlos.

Como el crudo no da para practicar limpieza, el resto del laboratorio usa el dataset
**físicamente coherente** que genera el simulador del Sitio 3, que sí trae defectos
realistas (sensores pegados, derivas) y, sobre todo, **sesgo de claves**, que es lo que
hace interesante al shuffling.

## 4. El dataset del simulador: formato largo y columnar

`salida_sitio3/telemetria.parquet` está en **Parquet particionado por día**. Frente al JSONL
del crudo tiene tres ventajas que se notan de inmediato:

1. **Columnar**: si pides 2 de 9 columnas, lee solo esas 2 del disco.
2. **Tipado**: el esquema viaja dentro del archivo, no hay que inferir ni castear.
3. **Particionado**: filtrar por `fecha` descarta directorios completos sin abrirlos
   (*partition pruning*).

In [8]:
RUTA_TELEMETRIA = "/opt/workspace/salida_sitio3/telemetria.parquet"
RUTA_VERDAD     = "/opt/workspace/salida_sitio3/verdad_terreno.parquet"

tele = spark.read.parquet(RUTA_TELEMETRIA)
tele.printSchema()
print(f"Filas: {tele.count():,}   Particiones: {tele.rdd.getNumPartitions()}")


AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/opt/workspace/salida_sitio3/telemetria.parquet. SQLSTATE: 42K03

In [13]:
# --- Partition pruning: Spark ni siquiera abre los días que no pides ---
un_dia = tele.filter(F.col("fecha") == "2026-01-14")

print("Plan físico de la consulta filtrada:\n")
plan = un_dia._jdf.queryExecution().executedPlan().toString()
# Se busca la línea donde Spark declara cuántas particiones leyó de verdad
for linea in plan.splitlines():
    if "PartitionFilters" in linea or "FileScan" in linea:
        print(linea.strip()[:300])
        break

print(f"\nFilas de ese día: {un_dia.count():,}  de  {tele.count():,} totales")
print("Fíjate en PartitionFilters: ahí está la prueba de que descartó los otros días")
print("sin leerlos. Si particionas por una columna que nunca filtras, no sirve de nada.")


Plan físico de la consulta filtrada:

+- FileScan parquet [timestamp_utc#167,entidad_id#168,entidad_tipo#169,alimentador_id#170,variable#171,valor#172,unidad#173,calidad_sensor#174,fecha#175] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/opt/workspace/salida_sitio3/telemetria.parquet], Parti

Filas de ese día: 1,794,054  de  12,790,650 totales
Fíjate en PartitionFilters: ahí está la prueba de que descartó los otros días
sin leerlos. Si particionas por una columna que nunca filtras, no sirve de nada.


## 5. Limpieza: lo que de verdad está sucio

El simulador separa el **estado físico verdadero** de la **lectura del sensor**. La columna
`calidad_sensor` marca cuándo el instrumento está fallando, y eso es una anomalía de
*instrumento*, no de *proceso*. Distinguirlas es justamente lo que la Fase 3 tendrá que
aprender a hacer.

In [14]:
tele.groupBy("calidad_sensor").count().orderBy(F.desc("count")).show(truncate=False)

print("ok           -> lectura confiable")
print("asentamiento -> la báscula aún oscila; el valor todavía no es el peso final")
print("deriva       -> el sensor acumula un offset creciente: el proceso está bien")
print("pegado       -> el sensor repite el último valor: el proceso puede estar cambiando")


[Stage 30:>                                                         (0 + 4) / 4]

+--------------+--------+
|calidad_sensor|count   |
+--------------+--------+
|ok            |12542212|
|asentamiento  |246482  |
|deriva        |1214    |
|pegado        |742     |
+--------------+--------+

ok           -> lectura confiable
asentamiento -> la báscula aún oscila; el valor todavía no es el peso final
deriva       -> el sensor acumula un offset creciente: el proceso está bien
pegado       -> el sensor repite el último valor: el proceso puede estar cambiando


In [15]:
# Rangos físicamente admisibles por variable. Fuera de esto, es un error de medición
# (no un dato interesante). Es una tabla pequeña: la usaremos también para el broadcast join.
rangos = spark.createDataFrame([
    ("temperatura_c",        -40.0,   60.0, "C"),
    ("potencia_kw",          -10.0,  100.0, "kW"),
    ("tension_v",              0.0,  500.0, "V"),
    ("demanda_kw",             0.0, 1000.0, "kW"),
    ("velocidad_ms",           0.0,   10.0, "m/s"),
    ("corriente_a",            0.0,  400.0, "A"),
    ("temperatura_polines_c", -20.0,  120.0, "C"),
    ("tonelaje_cola_th",       0.0, 2000.0, "t/h"),
    ("tonelaje_cabeza_th",     0.0, 2000.0, "t/h"),
    ("viento_ms",              0.0,   60.0, "m/s"),
    ("humedad_pct",            0.0,  100.0, "%"),
], ["variable", "min_valido", "max_valido", "unidad_esperada"])

rangos.show(5, False)
print(f"Tabla de rangos: {rangos.count()} filas. Es una DIMENSIÓN (chica),")
print("frente a la telemetría que es un HECHO (enorme). Recuérdalo para el bloque 9.")


+-------------+----------+----------+---------------+
|variable     |min_valido|max_valido|unidad_esperada|
+-------------+----------+----------+---------------+
|temperatura_c|-40.0     |60.0      |C              |
|potencia_kw  |-10.0     |100.0     |kW             |
|tension_v    |0.0       |500.0     |V              |
|demanda_kw   |0.0       |1000.0    |kW             |
|velocidad_ms |0.0       |10.0      |m/s            |
+-------------+----------+----------+---------------+
only showing top 5 rows
Tabla de rangos: 11 filas. Es una DIMENSIÓN (chica),
frente a la telemetría que es un HECHO (enorme). Recuérdalo para el bloque 9.


In [16]:
# --- Capa limpia ("silver") ---------------------------------------------------
# Se conservan las filas con sensor degradado pero MARCADAS, no se borran: para
# entrenar un detector de anomalías hacen falta los ejemplos malos.
limpio = (
    tele
    # 1) tipado explícito del valor
    .withColumn("valor", F.col("valor").cast("double"))
    # 2) descartar lecturas sin valor numérico
    .filter(F.col("valor").isNotNull())
    # 3) marcar (no borrar) las lecturas de instrumento degradado
    .withColumn("sensor_confiable", F.col("calidad_sensor") == "ok")
    # 4) columnas de tiempo útiles para agrupar después
    .withColumn("hora", F.hour("timestamp_utc"))
)

antes, despues = tele.count(), limpio.count()
print(f"Filas antes   : {antes:,}")
print(f"Filas después : {despues:,}   (descartadas: {antes - despues:,})")
limpio.select("timestamp_utc","entidad_id","variable","valor","calidad_sensor",
              "sensor_confiable").show(5, False)


Filas antes   : 12,790,650
Filas después : 12,790,650   (descartadas: 0)
+-------------------+----------+---------------------+------------------+--------------+----------------+
|timestamp_utc      |entidad_id|variable             |valor             |calidad_sensor|sensor_confiable|
+-------------------+----------+---------------------+------------------+--------------+----------------+
|2026-01-13 00:00:00|grua_1    |fase_actual          |4.0               |ok            |true            |
|2026-01-13 00:00:00|grua_1    |potencia_kw          |-5.0              |ok            |true            |
|2026-01-13 00:00:00|faja_1    |velocidad_ms         |4.2               |ok            |true            |
|2026-01-13 00:00:00|faja_1    |corriente_a          |177.63304082892006|ok            |true            |
|2026-01-13 00:00:00|faja_1    |temperatura_polines_c|31.679513488495473|ok            |true            |
+-------------------+----------+---------------------+------------------+------

## 6. Shuffling: el cuello de botella

Una operación **estrecha** (`filter`, `withColumn`) transforma cada partición por separado:
no hay comunicación entre nodos.

Una operación **ancha** (`groupBy`, `join`, `distinct`, `orderBy`) necesita que todas las
filas con la misma clave terminen en el mismo sitio. Eso obliga a **redistribuir los datos
por la red**: eso es el *shuffle*.

El shuffle escribe a disco, viaja por red y vuelve a leer. Es, con enorme diferencia, la
parte más cara de cualquier job de Spark. Todo lo que sigue es aprender a que duela menos.

In [17]:
# Mirar el plan es la herramienta número uno para detectar shuffles.
# Cada "Exchange" en el plan físico es un shuffle.
consulta = tele.groupBy("entidad_id", "variable").agg(F.avg("valor"))

plan = consulta._jdf.queryExecution().executedPlan().toString()
print("Etapas del plan físico (busca 'Exchange'):\n")
for linea in plan.splitlines()[:14]:
    marca = "  <-- SHUFFLE" if "Exchange" in linea else ""
    print(linea[:110] + marca)


Etapas del plan físico (busca 'Exchange'):

AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[entidad_id#168, variable#171], functions=[avg(valor#172)], output=[entidad_id#168, vari
   +- Exchange hashpartitioning(entidad_id#168, variable#171, 200), ENSURE_REQUIREMENTS, [plan_id=634]  <-- SHUFFLE
      +- HashAggregate(keys=[entidad_id#168, variable#171], functions=[partial_avg(valor#172)], output=[entida
         +- Project [entidad_id#168, variable#171, valor#172]
            +- FileScan parquet [entidad_id#168,variable#171,valor#172,fecha#175] Batched: true, DataFilters: 


### 6.1 El default de 200 particiones

`spark.sql.shuffle.partitions` vale **200** por defecto. Ese número viene de clústeres
grandes de hace una década y **no se adapta a tu máquina**.

En este clúster tenemos 4 cores y la consulta produce ~329 grupos. Con 200 particiones de
shuffle, Spark crea 200 tareas que 4 cores tienen que despachar de a 4. Cada tarea tiene un
costo fijo de arranque que no depende de cuántos datos procese: con particiones diminutas,
**todo el tiempo se va en administrar tareas, no en calcular**.

In [18]:
tele.cache(); tele.count()   # se cachea para que midamos el shuffle, no la lectura de disco

def medir(etiqueta, particiones, aqe):
    spark.conf.set("spark.sql.adaptive.enabled", aqe)
    spark.conf.set("spark.sql.shuffle.partitions", particiones)
    ini = time.time()
    r = tele.groupBy("entidad_id", "variable").agg(
        F.avg("valor").alias("prom"), F.count("*").alias("n"))
    salida = r.rdd.getNumPartitions()
    grupos = r.count()
    print(f"{etiqueta:<28} {time.time()-ini:6.2f} s   "
          f"particiones_salida={salida:<4} grupos={grupos}")

print(f"{'configuración':<28} {'tiempo':>8}\n" + "-"*62)
medir("200 (default), sin AQE", 200, False)
medir("8 (= 2x cores), sin AQE",  8, False)
medir("200, CON AQE",           200, True)


configuración                  tiempo
--------------------------------------------------------------


200 (default), sin AQE         4.03 s   particiones_salida=200  grupos=329


8 (= 2x cores), sin AQE        1.43 s   particiones_salida=8    grupos=329


[Stage 56:>                                                         (0 + 4) / 4]

200, CON AQE                   2.91 s   particiones_salida=1    grupos=329


### Cómo leer ese resultado

- **200 sin AQE** es el peor caso: 200 tareas para 329 grupos que caben en 8 particiones.
- **8 sin AQE** es el ajuste manual. La regla práctica es **2 a 4 veces el número de cores**
  del clúster para datos de este tamaño.
- **200 con AQE** es el punto intermedio: *Adaptive Query Execution* mira las estadísticas
  reales al terminar la etapa y **fusiona** las particiones pequeñas. Fíjate en que las
  particiones de salida se colapsan solas.

> **AQE te salva, pero conviene saber de qué te está salvando.** Está activado por defecto
> en `docker/spark-defaults.conf`. Si lo apagas, vuelves al comportamiento de arriba.

## 7. Sesgo de claves (*data skew*)

Ajustar el número de particiones no sirve de nada si **los datos están mal repartidos**.

En este dataset, `faja_1` se muestrea cada segundo y los reefers cada 60 s. Resultado:
una sola entidad concentra casi la mitad de todas las filas. Al agrupar por `entidad_id`,
la partición que recibe `faja_1` tarda ~100 veces más que las demás, y **la etapa completa
termina cuando termina la tarea más lenta**. Los otros 3 cores esperan de brazos cruzados.

In [19]:
conteo = tele.groupBy("entidad_id").count().orderBy(F.desc("count"))
total = tele.count()
top = conteo.limit(5).collect()

print(f"{'entidad':<14}{'filas':>12}{'% del total':>14}")
print("-" * 40)
for r in top:
    print(f"{r['entidad_id']:<14}{r['count']:>12,}{100*r['count']/total:>13.1f}%")

mayor = top[0]["count"]
menor = conteo.orderBy("count").first()["count"]
print(f"\nRatio entre la clave mayor y la menor: {mayor/menor:.0f} a 1")
print("Cualquier ratio por encima de ~10:1 ya produce tareas rezagadas visibles.")


entidad              filas   % del total
----------------------------------------
faja_1           6,048,000         47.3%
bascula_1          272,184          2.1%
ambiente_1         100,800          0.8%
reefer_69           60,480          0.5%
reefer_2            60,480          0.5%

Ratio entre la clave mayor y la menor: 300 a 1
Cualquier ratio por encima de ~10:1 ya produce tareas rezagadas visibles.


### Mitigación: *salting*

La idea es **partir artificialmente la clave caliente** en `n` subclaves (`faja_1#0`,
`faja_1#1`, …) para que el trabajo se reparta, agregar sobre las subclaves, y **recombinar
al final**.

Se paga con una agregación extra. Solo vale la pena cuando el sesgo es grande — como aquí.

In [20]:
N_SAL = 8   # en cuántos trozos se parte la clave caliente

# 1) Se añade sal SOLO a la entidad caliente; el resto no se toca.
con_sal = tele.withColumn(
    "clave_salada",
    F.when(F.col("entidad_id") == "faja_1",
           F.concat(F.col("entidad_id"), F.lit("#"),
                    (F.rand(seed=7) * N_SAL).cast("int").cast("string")))
     .otherwise(F.col("entidad_id"))
)

spark.conf.set("spark.sql.adaptive.enabled", False)   # apagado para ver el efecto puro
spark.conf.set("spark.sql.shuffle.partitions", 8)

ini = time.time()
sin_salting = tele.groupBy("entidad_id").agg(F.avg("valor").alias("prom")).count()
t_sin = time.time() - ini

ini = time.time()
parcial = con_sal.groupBy("clave_salada").agg(
    F.sum("valor").alias("suma"), F.count("*").alias("n"))
final = (parcial
         .withColumn("entidad_id", F.split(F.col("clave_salada"), "#").getItem(0))
         .groupBy("entidad_id")
         .agg((F.sum("suma") / F.sum("n")).alias("prom")))
_ = final.count()
t_con = time.time() - ini

print(f"groupBy directo (sesgado) : {t_sin:6.2f} s")
print(f"con salting en 2 etapas   : {t_con:6.2f} s")
print("\nOJO: con un dataset de este tamaño el salting puede incluso salir más lento,")
print("porque añade una agregación completa. La técnica rinde cuando la clave caliente")
print("no cabe en memoria de un executor. Lo importante es que sepas reconocer el")
print("síntoma en la UI: una tarea que tarda muchísimo más que sus hermanas.")

spark.conf.set("spark.sql.adaptive.enabled", True)   # se vuelve a dejar como estaba


[Stage 74:===========================================>              (3 + 1) / 4]

groupBy directo (sesgado) :   0.82 s
con salting en 2 etapas   :   1.91 s

OJO: con un dataset de este tamaño el salting puede incluso salir más lento,
porque añade una agregación completa. La técnica rinde cuando la clave caliente
no cabe en memoria de un executor. Lo importante es que sepas reconocer el
síntoma en la UI: una tarea que tarda muchísimo más que sus hermanas.


## 8. Joins: *broadcast* vs *sort-merge*

Un `join` normal (**sort-merge**) hace shuffle de **las dos** tablas: ordena y redistribuye
ambas por la clave. Si una de las dos es pequeña, eso es un desperdicio enorme.

Un **broadcast join** copia la tabla chica entera a la memoria de cada executor y evita el
shuffle por completo. Nuestra `verdad_terreno` tiene **9 filas**: es el caso perfecto.

In [21]:
verdad = spark.read.parquet(RUTA_VERDAD)
print(f"verdad_terreno: {verdad.count()} filas  <- diminuta")
verdad.select("falla_id","tipo_falla","entidad_afectada","severidad").show(5, False)

# Spark hace broadcast automático si la tabla es menor que este umbral (10 MB por defecto).
umbral = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
print(f"\nUmbral de broadcast automático: {umbral} bytes")


verdad_terreno: 9 filas  <- diminuta
+--------+--------------------+----------------+---------+
|falla_id|tipo_falla          |entidad_afectada|severidad|
+--------+--------------------+----------------+---------+
|F006    |rodamiento_faja     |faja_1          |severa   |
|F001    |compresor_degradado |reefer_51       |moderada |
|F004    |apertura_alimentador|alimentador_2   |severa   |
|F002    |puerta_mal_cerrada  |reefer_50       |moderada |
|F009    |deriva_sensor       |reefer_20       |leve     |
+--------+--------------------+----------------+---------+
only showing top 5 rows

Umbral de broadcast automático: 10485760b bytes


In [22]:
etiquetas = verdad.select(
    F.col("entidad_afectada").alias("entidad_id"),
    F.col("tipo_falla"), F.col("severidad"),
    F.col("t_inicio"), F.col("t_fin"))

# --- sin broadcast: se fuerza sort-merge para ver la diferencia en el plan ---
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
plan_smj = (tele.join(etiquetas, "entidad_id", "left")
            ._jdf.queryExecution().executedPlan().toString())

# --- con broadcast explícito ---
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)
plan_bhj = (tele.join(F.broadcast(etiquetas), "entidad_id", "left")
            ._jdf.queryExecution().executedPlan().toString())

def resumen(plan, titulo):
    print(f"--- {titulo} ---")
    print(f"  Exchange (shuffles) : {plan.count('Exchange')}")
    print(f"  Sort                : {plan.count('Sort ')}")
    print(f"  BroadcastHashJoin   : {plan.count('BroadcastHashJoin')}")
    print(f"  SortMergeJoin       : {plan.count('SortMergeJoin')}")

resumen(plan_smj, "SIN broadcast")
resumen(plan_bhj, "CON broadcast")
print("\nEl broadcast elimina el shuffle Y el ordenamiento de la tabla grande.")


--- SIN broadcast ---
  Exchange (shuffles) : 2
  Sort                : 2
  BroadcastHashJoin   : 0
  SortMergeJoin       : 1
--- CON broadcast ---
  Exchange (shuffles) : 1
  Sort                : 0
  BroadcastHashJoin   : 1
  SortMergeJoin       : 0

El broadcast elimina el shuffle Y el ordenamiento de la tabla grande.


### 8.1 La trampa del *fan-out*: un join que multiplica tus filas

Aquí viene el error más silencioso de todo el pipeline, y conviene cometerlo en clase antes
de cometerlo en producción.

Un `left join` **no conserva** el número de filas de la tabla izquierda. Conserva *como
mínimo* ese número: si una clave aparece **N veces** en la tabla derecha, cada fila de la
izquierda se duplica **N veces**.

`faja_1` sufrió dos fallas distintas (`rodamiento_faja` y `sobrecarga_faja`), así que
aparece **dos veces** en la verdad de terreno. Vamos a medir el daño.

In [23]:
n_original = tele.count()
n_join_ingenuo = tele.join(F.broadcast(etiquetas), "entidad_id", "left").count()

print(f"Filas antes del join : {n_original:,}")
print(f"Filas después        : {n_join_ingenuo:,}")
print(f"Filas fabricadas     : {n_join_ingenuo - n_original:,}")
print()
print("Claves duplicadas en la tabla de etiquetas:")
(etiquetas.groupBy("entidad_id").count()
 .filter("count > 1").orderBy(F.desc("count")).show(5, False))
print("Comprueba SIEMPRE el conteo de filas antes y después de un join.")


Filas antes del join : 12,790,650
Filas después        : 18,838,650
Filas fabricadas     : 6,048,000

Claves duplicadas en la tabla de etiquetas:
+----------+-----+
|entidad_id|count|
+----------+-----+
|faja_1    |2    |
+----------+-----+

Comprueba SIEMPRE el conteo de filas antes y después de un join.


### 8.2 El segundo error: ignorar el tiempo

Aunque arregláramos la duplicación, el join seguiría mal. `faja_1` tiene una falla de
rodamiento, sí — **pero solo durante unas horas concretas**. Unir por `entidad_id` a secas
etiqueta como anómalas los 6 millones de lecturas de la faja, incluidas las de los días en
que funcionaba perfecta.

Un detector entrenado con esas etiquetas aprendería que "faja_1 = anomalía", que es
exactamente la conclusión inútil que queremos evitar.

La unión correcta es un **range join**: coincide la entidad **y además** el instante de la
lectura cae dentro de la ventana de la falla.

In [24]:
# Range join: la condición incluye el intervalo temporal de la falla.
cond = (
    (tele["entidad_id"] == etiquetas["entidad_id"]) &
    (tele["timestamp_utc"] >= etiquetas["t_inicio"]) &
    (tele["timestamp_utc"] <= etiquetas["t_fin"])
)

etiquetada = (
    tele.join(F.broadcast(etiquetas), cond, "left")
        .drop(etiquetas["entidad_id"])
)

n_correcto = etiquetada.count()
print(f"Filas tras el range join : {n_correcto:,}")
print(f"Filas originales         : {n_original:,}")

# La aserción es la red de seguridad: si algún día dos fallas se solapan sobre la
# misma entidad, el conteo crecerá y queremos enterarnos AQUÍ, no en la Fase 3.
assert n_correcto == n_original, (
    f"El join alteró el número de filas ({n_original:,} -> {n_correcto:,}). "
    "Hay fallas solapadas sobre la misma entidad: decide si te quedas con la más "
    "severa o si emites una fila por falla."
)
print("\nConteo intacto.")

etiquetada.filter(F.col("tipo_falla").isNotNull()).groupBy(
    "entidad_id", "tipo_falla", "severidad").count().orderBy(F.desc("count")).show(10, False)


Filas tras el range join : 12,790,650
Filas originales         : 12,790,650

Conteo intacto.


[Stage 97:==============>                                           (1 + 3) / 4]

+-------------+--------------------+---------+------+
|entidad_id   |tipo_falla          |severidad|count |
+-------------+--------------------+---------+------+
|faja_1       |rodamiento_faja     |severa   |829860|
|faja_1       |sobrecarga_faja     |moderada |18920 |
|reefer_51    |compresor_degradado |moderada |3966  |
|reefer_20    |deriva_sensor       |leve     |3642  |
|reefer_3     |sensor_pegado       |moderada |2226  |
|reefer_50    |puerta_mal_cerrada  |moderada |1902  |
|reefer_36    |desconexion_equipo  |severa   |678   |
|alimentador_2|apertura_alimentador|severa   |292   |
+-------------+--------------------+---------+------+



Ahora los conteos tienen sentido: solo las lecturas **dentro de la ventana** de cada
falla quedan etiquetadas, y el total de filas no cambió.

> **Nota sobre los alimentadores.** Las fallas de red aparecen con
> `entidad_afectada = "alimentador_2"`, así que este join solo etiqueta las lecturas del
> propio alimentador. Pero un corte de alimentador afecta físicamente a **todos los reefers
> colgados de él**, que en la telemetría llevan `alimentador_id = 2`.
>
> **Ejercicio 6.** Amplía el join para propagar las fallas de red a los reefers de ese
> alimentador usando `alimentador_id`. Cuida que no se te dispare el fan-out.

## 9. `repartition` vs `coalesce`, y `cache`

Dos operaciones que parecen lo mismo y no lo son:

| | ¿Hace shuffle? | ¿Para qué sirve? |
|---|---|---|
| `repartition(n)` | **Sí**, redistribuye todo | Subir particiones o reequilibrar datos sesgados |
| `coalesce(n)` | **No**, solo fusiona vecinas | Bajar particiones antes de escribir, barato |

Regla: **para reducir, `coalesce`; para reequilibrar, `repartition`.** Usar `repartition`
solo para bajar de 200 a 8 particiones paga un shuffle completo sin necesidad.

In [25]:
print(f"Particiones originales     : {tele.rdd.getNumPartitions()}")
print(f"Tras coalesce(2)           : {tele.coalesce(2).rdd.getNumPartitions()}   (sin shuffle)")
print(f"Tras repartition(16)       : {tele.repartition(16).rdd.getNumPartitions()}  (con shuffle)")

print("\n--- cache ---")
sin_cache = spark.read.parquet(RUTA_TELEMETRIA)
ini = time.time(); sin_cache.count(); t1 = time.time() - ini
ini = time.time(); sin_cache.count(); t2 = time.time() - ini
sin_cache.cache(); sin_cache.count()
ini = time.time(); sin_cache.count(); t3 = time.time() - ini

print(f"1ª lectura desde disco     : {t1:.2f} s")
print(f"2ª lectura desde disco     : {t2:.2f} s")
print(f"Lectura ya cacheada        : {t3:.2f} s")
print("\ncache() vale la pena cuando el MISMO DataFrame se usa varias veces.")
print("Cachear algo que se usa una sola vez solo gasta memoria de los executors.")
sin_cache.unpersist()


Particiones originales     : 4
Tras coalesce(2)           : 2   (sin shuffle)


[Stage 100:==========================================>              (3 + 1) / 4]

Tras repartition(16)       : 16  (con shuffle)

--- cache ---


1ª lectura desde disco     : 0.13 s
2ª lectura desde disco     : 0.13 s
Lectura ya cacheada        : 0.09 s

cache() vale la pena cuando el MISMO DataFrame se usa varias veces.
Cachear algo que se usa una sola vez solo gasta memoria de los executors.


DataFrame[timestamp_utc: timestamp, entidad_id: string, entidad_tipo: string, alimentador_id: smallint, variable: string, valor: double, unidad: string, calidad_sensor: string, fecha: date]

## 10. Spark SQL: la misma consulta, otra sintaxis

DataFrames y SQL compilan **exactamente al mismo plan**. Usa el que te resulte más legible;
no hay penalización de rendimiento por elegir uno u otro.

In [26]:
tele.createOrReplaceTempView("telemetria")
verdad.createOrReplaceTempView("verdad_terreno")

spark.sql("""
    SELECT  t.entidad_tipo,
            t.variable,
            COUNT(*)                                   AS lecturas,
            ROUND(AVG(t.valor), 2)                     AS promedio,
            SUM(CASE WHEN t.calidad_sensor <> 'ok'
                     THEN 1 ELSE 0 END)                AS lecturas_degradadas
    FROM    telemetria t
    WHERE   t.entidad_tipo IN ('reefer', 'faja')
    GROUP BY t.entidad_tipo, t.variable
    ORDER BY lecturas DESC
""").show(10, False)


[Stage 114:==========================================>              (3 + 1) / 4]

+------------+---------------------+--------+--------+-------------------+
|entidad_tipo|variable             |lecturas|promedio|lecturas_degradadas|
+------------+---------------------+--------+--------+-------------------+
|reefer      |temperatura_c        |2016000 |-18.0   |1956               |
|reefer      |potencia_kw          |2016000 |1.51    |0                  |
|reefer      |compresor_on         |2016000 |0.18    |0                  |
|faja        |corriente_a          |1209600 |117.15  |0                  |
|faja        |velocidad_ms         |1209600 |1.99    |0                  |
|faja        |temperatura_polines_c|1209600 |25.34   |0                  |
|faja        |tonelaje_cabeza_th   |1209600 |265.98  |0                  |
|faja        |tonelaje_cola_th     |1209600 |265.98  |0                  |
+------------+---------------------+--------+--------+-------------------+



## 11. Escribir la capa limpia

Última decisión de diseño, y la que más impacto tiene en quien consuma estos datos después:
**por qué columna particionar**.

- Particiona por algo que **se filtre a menudo** (aquí: `fecha`).
- Que tenga **cardinalidad moderada**. Particionar por `timestamp_utc` crearía millones de
  directorios con un archivo minúsculo cada uno: es el *small files problem*, y hunde
  cualquier sistema de archivos distribuido.
- `coalesce` antes de escribir para no generar 200 archivos diminutos.

In [27]:
RUTA_SALIDA = "/opt/workspace/salida_fase2/telemetria_limpia.parquet"

cond_limpio = (
    (limpio["entidad_id"] == etiquetas["entidad_id"]) &
    (limpio["timestamp_utc"] >= etiquetas["t_inicio"]) &
    (limpio["timestamp_utc"] <= etiquetas["t_fin"])
)

salida = (
    limpio
    .join(F.broadcast(etiquetas), cond_limpio, "left")   # range join, no por clave suelta
    .drop(etiquetas["entidad_id"])
    .withColumn("es_anomalia_conocida", F.col("tipo_falla").isNotNull())
    .select("timestamp_utc", "fecha", "hora", "entidad_id", "entidad_tipo",
            "alimentador_id", "variable", "valor", "unidad",
            "calidad_sensor", "sensor_confiable", "tipo_falla", "severidad",
            "es_anomalia_conocida")
)

ini = time.time()
(salida
 .coalesce(4)                       # 4 archivos por día, no 200
 .write.mode("overwrite")
 .partitionBy("fecha")
 .parquet(RUTA_SALIDA))

print(f"Escrito en {time.time()-ini:.1f} s  ->  {RUTA_SALIDA}")


Escrito en 18.0 s  ->  /opt/workspace/salida_fase2/telemetria_limpia.parquet


In [28]:
# --- Verificación: releer lo escrito y comprobar que cuadra ---
verif = spark.read.parquet(RUTA_SALIDA)
n_esc, n_esp = verif.count(), limpio.count()
print(f"Filas escritas : {n_esc:,}")
print(f"Filas esperadas: {n_esp:,}")
assert n_esc == n_esp, f"La capa limpia cambió de tamaño: {n_esp:,} -> {n_esc:,}"
print(f"Días           : {verif.select('fecha').distinct().count()}")
print()
verif.groupBy("es_anomalia_conocida").count().show()
verif.select("entidad_id","variable","valor","calidad_sensor",
             "tipo_falla","severidad").filter("es_anomalia_conocida").show(5, False)


Filas escritas : 12,790,650
Filas esperadas: 12,790,650
Días           : 7



+--------------------+--------+
|es_anomalia_conocida|   count|
+--------------------+--------+
|                true|  861486|
|               false|11929164|
+--------------------+--------+

+----------+---------------------+------------------+--------------+---------------+---------+
|entidad_id|variable             |valor             |calidad_sensor|tipo_falla     |severidad|
+----------+---------------------+------------------+--------------+---------------+---------+
|faja_1    |velocidad_ms         |4.2               |ok            |rodamiento_faja|severa   |
|faja_1    |corriente_a          |177.63304082892006|ok            |rodamiento_faja|severa   |
|faja_1    |temperatura_polines_c|31.679513488495473|ok            |rodamiento_faja|severa   |
|faja_1    |tonelaje_cola_th     |548.9541905349603 |ok            |rodamiento_faja|severa   |
|faja_1    |tonelaje_cabeza_th   |563.9319910602381 |ok            |rodamiento_faja|severa   |
+----------+---------------------+-------------

## Cierre

Lo que queda escrito en `salida_fase2/telemetria_limpia.parquet` es el insumo de la Fase 3:
telemetría tipada, con la calidad del sensor marcada y **etiquetada con la verdad de
terreno**, particionada por día.

### Lo que hay que llevarse de esta fase

1. **El esquema se declara, no se adivina.** `inferSchema` cuesta una pasada completa.
2. **Perfila antes de limpiar.** Limpiar a ciegas borra datos buenos y deja pasar los malos.
3. **Todo shuffle es un `Exchange` en el plan.** `explain()` es la herramienta de diagnóstico.
4. **200 particiones no es un número mágico**, es un default de otra época. Ajústalo o
   deja que AQE lo haga, pero sabiendo qué está pasando.
5. **El sesgo de claves no se arregla con más particiones.** Se reconoce por una tarea
   rezagada y se ataca con salting o con AQE skew join.
6. **Si una tabla es chica, hazle broadcast.** Ahorra el shuffle de la tabla grande.
7. **Parquet particionado > JSON**, en lectura, en tamaño y en tipos.

### Ejercicios

> **2.** Vuelve a correr el bloque 6 con `spark.sql.shuffle.partitions` en 1, 4, 8, 32, 200 y
> grafica tiempo contra particiones. ¿Dónde está el mínimo y por qué hay un mínimo?
>
> **3.** En <http://localhost:8080> entra a una aplicación terminada, busca la etapa del
> `groupBy` y compara la duración de la tarea más lenta con la mediana. Ese cociente es tu
> métrica de sesgo.
>
> **4.** Levanta un worker más (`docker compose up -d --scale spark-worker=3`) y repite la
> medición. ¿Se reduce el tiempo proporcionalmente? Si no, ¿qué lo limita?
>
> **5.** Sustituye el `broadcast` explícito por `spark.sql.autoBroadcastJoinThreshold = -1`
> y compara los planes. ¿Cuántos `Exchange` aparecen de más?

In [29]:
# Cerrar la sesión libera los executors para el resto del curso.
spark.stop()
print("Sesión cerrada. Los workers quedan libres.")


Sesión cerrada. Los workers quedan libres.
